In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import fiona
from fiona import transform
import gdal
import rasterio as rio
from rasterio import mask
from tqdm.notebook import tqdm

from neuralhydrology.datasetzoo.camelsus import load_camels_us_attributes

project_root = Path('/publicwork/gauch/GRIP-GL/scripts/MachineLearning')
data_root = Path('/publicdata/Hydrology/')

ModuleNotFoundError: No module named 'gdal'

In [ ]:
shape_sets = {'camels': ('camels.geojson', 'hru_id'), 'canopex': ('canopex_less15000km2_lower53deg_nldasoverlapattribute.geojson', 'ID')}

for shape_set, (file, id_col) in shape_sets.items():
    i = 0
    with fiona.open(project_root / 'shapefiles' / file) as shapefile:
        shape_set_crs[shape_set] = shapefile.crs
        for shape in shapefile:
            if shape_set == 'canopex' and shape['properties']['ee_nldas_extent_pc'] < 99.99:
                continue  # only use basins that are fully inside of NLDAS
            shapes[(str(shape['properties'][id_col]), shape_set)] = shape['geometry']
            i += 1
    print(f'found {i} {shape_set} basins')

In [2]:
shapes = {}
shapefile_crs = None

for typ in ['calibration', 'validation']:
    gl_shapefiles = Path(project_root / f'../../data/shapefiles/great-lakes/{typ}/').glob('*/*.shp')
    i = 0
    for gl_shp in gl_shapefiles:
        basin_id = gl_shp.parent.stem
        if basin_id in shapes:
            print(f'Already processed {basin_id}')
        with fiona.open(gl_shp) as shapefile:
            shapefile_crs = shapefile.crs
            if len(shapefile) != 1:
                raise ValueError(f'{gl_shp} contains != 1 shapes')
            shapes[basin_id] = shapefile[0]['geometry']
            i += 1
    print(f'found {i} {typ} basins')

found 141 calibration basins
found 71 validation basins


## Landcover

In [ ]:
lc_classes = {
    1: 'Temperate-or-sub-polar-needleleaf-forest',
    2: 'Sub-polar-taiga-needleleaf-forest',
    3: 'Tropical-or-sub-tropical-broadleaf-evergreen-forest',
    4: 'Tropical-or-sub-tropical-broadleaf-deciduous-forest',
    5: 'Temperate-or-sub-polar-broadleaf-deciduous-forest',
    6: 'Mixed-Forest',
    7: 'Tropical-or-sub-tropical-shrubland',
    8: 'Temperate-or-sub-polar-shrubland',
    9: 'Tropical-or-sub-tropical-grassland',
    10: 'Temperate-or-sub-polar-grassland',
    11: 'Sub-polar-or-polar-shrubland-lichen-moss',
    12: 'Sub-polar-or-polar-grassland-lichen-moss',
    13: 'Sub-polar-or-polar-barren-lichen-moss',
    14: 'Wetland',
    15: 'Cropland',
    16: 'Barren-Lands',
    17: 'Urban-and-Built-up',
    18: 'Water',
    19: 'Snow-and-Ice',
}
lc_fractions = pd.DataFrame(columns=lc_classes.values(), index=shapes.keys())

with rio.open(project_root / 'gridded' / 'landcover' / 'NA_NALCMS_2010_v2_land_cover_30m/NA_NALCMS_2010_v2_land_cover_30m.tif') as gridded_ds:
    for basin, shape in tqdm(shapes.items()):
        
        # transform shape to gridded_ds crs
        transformed_shape = transform.transform_geom(shapefile_crs, gridded_ds.crs.data, shape)
        
        # crop to basin outline
        cropped_ds, _ = mask.mask(gridded_ds, [transformed_shape], crop=True,
                                  filled=True, nodata=gridded_ds.nodata)
        cropped_ds = cropped_ds.astype(np.float)
        cropped_ds[cropped_ds == gridded_ds.nodata] = np.nan
        
        for lc_id, lc_class in lc_classes.items():
            lc_fractions.loc[basin, lc_class] = (cropped_ds == lc_id).sum() / (~np.isnan(cropped_ds)).sum()

## Soil

In [ ]:
soil_sets = ['BD', 'CLAY', 'GRAV', 'OC', 'SAND', 'SILT']
soil_data = pd.DataFrame(columns=soil_sets, index=shapes.keys())

for basin, shape in tqdm(shapes.items()):
    # transform shape to gridded_ds crs
    transformed_shape = transform.transform_geom(shapefile_crs, {'init': 'epsg:4326'}, shape)

    for soil_set in soil_sets:

        # soil data is split in two nc files, one containing the first 4 soil layers the other the second 4.
        cropped = []
        for i in [1, 2]:
            with rio.open(project_root / 'gridded' / 'soil' / f'{soil_set}{i}.nc') as gridded_ds:
                    # crop to basin outline
                    cropped_ds, _ = mask.mask(gridded_ds, [transformed_shape], crop=True,
                                              filled=True, nodata=gridded_ds.nodata)
                    cropped_ds = cropped_ds.astype(np.float)
                    cropped_ds[cropped_ds == gridded_ds.nodata] = np.nan
                    cropped.append(cropped_ds)

        cropped = np.concatenate(cropped, axis=0)
        soil_data.loc[basin, soil_set] = np.nanmean(cropped_ds)

## DEM

In [ ]:
dem_info = pd.DataFrame(columns=['mean_elev', 'mean_slope', 'std_elev', 'std_slope', 'area_km2'],
                        index=shapes.keys())

with rio.open(project_root / 'gridded' / 'dem' / 'na_ca_dem_3s.tif') as gridded_ds:
    for basin, shape in tqdm(shapes.items()):

        # transform shape to gridded_ds crs
        transformed_shape = transform.transform_geom(shapefile_crs, gridded_ds.crs.data, shape)

        # crop to basin outline
        cropped_ds, _ = mask.mask(gridded_ds, [transformed_shape], crop=True,
                                  filled=True, nodata=gridded_ds.nodata)
        cropped_ds = cropped_ds.astype(np.float)
        cropped_ds[cropped_ds == gridded_ds.nodata] = np.nan

        dem_info.loc[basin, 'mean_elev'] = np.nanmean(cropped_ds)
        dem_info.loc[basin, 'std_elev'] = np.nanstd(cropped_ds)

In [ ]:
# leave slope calculation to GDAL. Horizontal units are degrees, vertical units are meters, hence the scale factor (see https://gdal.org/programs/gdaldem.html)
!/system/apps/userenv/gauch/nh/bin/gdaldem slope -s 111120 gridded/dem/na_ca_dem_3s.tif /tmp/tmp-slope.tif

In [2]:
with rio.open('/tmp/tmp-slope.tif', driver='GTiff') as gridded_ds:
    for basin, shape in tqdm(shapes.items()):

        # transform shape to gridded_ds crs
        transformed_shape = transform.transform_geom(shapefile_crs, gridded_ds.crs.data, shape)

        # crop to basin outline
        cropped_ds, _ = mask.mask(gridded_ds, [transformed_shape], crop=True,
                                  filled=True, nodata=gridded_ds.nodata)
        cropped_ds = cropped_ds.astype(np.float)
        cropped_ds[cropped_ds == gridded_ds.nodata] = np.nan

        dem_info.loc[basin, 'mean_slope'] = np.nanmean(cropped_ds)
        dem_info.loc[basin, 'std_slope'] = np.nanstd(cropped_ds)

NameError: name 'rio' is not defined

In [8]:
!rm /tmp/tmp-slope.tif

In [9]:
for typ in ['calibration', 'validation']:
    gl_shapes = (project_root / f'../../data/shapefiles/great-lakes/{typ}/').glob('*/*.shp')
    for gl_shape in gl_shapes:
        basin = gl_shape.parent.stem
        df = gpd.read_file(gl_shape)
        df = df.to_crs('ESRI:102017')

        if len(df) != 1:
            raise ValueError(f'Found != 1 shapes in {gl_shape}')
        dem_info.loc[basin, 'area_km2'] = df.loc[0, 'geometry'].area * 1e-6

In [10]:
gl_metadata_obj1 = pd.read_csv(project_root / '../../data/objective_1/great-lakes/calibration/gauge_info.csv').set_index('ID')
gl_metadata_obj2 = pd.read_csv(project_root / '../../data/objective_2/great-lakes/calibration/gauge_info.csv').set_index('ID')
gl_metadata_obj1_val = pd.read_csv(project_root / '../../data/objective_1/great-lakes/validation/gauge_info.csv').set_index('ID')
gl_metadata_obj2_val = pd.read_csv(project_root / '../../data/objective_2/great-lakes/validation/gauge_info.csv').set_index('ID')
gl_metadata = gl_metadata_obj1.append(gl_metadata_obj2.loc[[station for station in gl_metadata_obj2.index if station not in gl_metadata_obj1.index]])
gl_metadata = gl_metadata.append(gl_metadata_obj1_val.loc[[station for station in gl_metadata_obj1_val.index if station not in gl_metadata.index]])
gl_metadata = gl_metadata.append(gl_metadata_obj2_val.loc[[station for station in gl_metadata_obj2_val.index if station not in gl_metadata.index]])

In [11]:
static_attrs = lc_fractions.join(dem_info).join(soil_data)
static_attrs.index.set_names('basin', inplace=True)

In [12]:
static_attrs['gauge_lat'] = np.nan
static_attrs['gauge_lon'] = np.nan
for basin in static_attrs.index:
    lat, lon = gl_metadata.loc[basin, ['Lat', 'Lon']]
    
    static_attrs.loc[basin, ['gauge_lat', 'gauge_lon']] = lat, lon

In [14]:
static_attrs.to_csv(project_root / 'attributes' / 'static_attributes.csv')

In [13]:
static_attrs

,Temperate-or-sub-polar-needleleaf-forest,Sub-polar-taiga-needleleaf-forest,Tropical-or-sub-tropical-broadleaf-evergreen-forest,Tropical-or-sub-tropical-broadleaf-deciduous-forest,Temperate-or-sub-polar-broadleaf-deciduous-forest,Mixed-Forest,Tropical-or-sub-tropical-shrubland,Temperate-or-sub-polar-shrubland,Tropical-or-sub-tropical-grassland,Temperate-or-sub-polar-grassland,...,std_slope,area_km2,BD,CLAY,GRAV,OC,SAND,SILT,gauge_lat,gauge_lon
basin,,,,,,,,,,,,,,,,,,,,,
04258000,0.168669,0,0,0,0.473207,0.0974367,0,0.0198054,0,0.00493554,...,3.11648,754.375,157.244,6.62932,23.3978,168.678,64.2905,28.1067,43.897222,-75.404167
02FE009,0.00134196,0,0,0,0.0871487,0.00718726,0,0.00255181,0,0.000106615,...,0.720807,388.311,156.111,25.8827,0,52.4312,30.4725,41.5599,43.684360,-81.541168
04072150,0.000258608,0,0,0,0.0534655,0.000891865,0,0.000696252,0,0.00178373,...,0.76031,271.462,176.934,29.475,11.258,290.934,39.0756,31.2375,44.533389,-88.129694
04166500,0.00389616,0,0,0,0.168655,0.0197328,0,0.000996604,0,0.00359342,...,1.16514,478.631,151.19,16.1888,12.3075,410.667,62.4993,19.9147,42.373092,-83.254651
04177000,0.00028364,0,0,0,0.0718974,0.00208181,0,6.15446e-05,0,0.0047095,...,0.715705,336.368,175.925,28.8021,5.13224,84.9648,41.5323,28.8441,41.659681,-83.612547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
04057510,0.109581,0,0,0,0.170004,0.128664,0,0.00574546,0,0.0146604,...,1.17731,515.668,110.52,5.49567,6.41869,1448.48,87.0507,7.06171,45.943023,-86.705700
04096015,0.00195482,0,0,0,0.150969,0.00873335,0,0.00138467,0,0.0151408,...,1.07538,198.864,169.736,21.5309,8.07818,75.1352,46.2948,31.5008,41.873654,-86.575022
02HF002,0.0169252,0,0,0,0.381735,0.386852,0,0.00394332,0,0.00347193,...,3.05127,1729.78,154.744,1.75738,1.75255,20.0009,88.9825,7.75993,44.731831,-78.818062
